# Question Classification with Sentence Embeddings — Codex-Style Embedding Model Comparison

1. Use the provided train.csv and test.csv files as-is  
2. Embed each question into a feature vector  
3. Learn the same kind of linear multiclass PyTorch model on top of each embedding space  
4. Compare the two embedding models head-to-head  
5. Identify where each embedding succeeds, where it fails, and which classes each model struggles with  
6. Use confusion matrices, disagreement examples, and coarse-category analysis to diagnose model deficiencies



In [ ]:
!pip install -q sentence-transformers torch scikit-learn pandas numpy matplotlib seaborn


In [ ]:
import copy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from sentence_transformers import SentenceTransformer
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

plt.rcParams["figure.figsize"] = (10, 6)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)


## Step 1: Load and inspect the data

In [ ]:
train_df = pd.read_csv('question_classificatrion/question_classification_dataset/train.csv')
test_df = pd.read_csv('question_classificatrion/question_classification_dataset/test.csv')

print("Provided train shape:", train_df.shape)
print("Provided test shape:", test_df.shape)
display(train_df.head())

print("\nNumber of coarse labels:", train_df["label-coarse"].nunique())
print("Number of fine labels:", train_df["label-fine"].nunique())

## Step 2: Create a holdout split from the provided training set

We keep the provided **test.csv** file as-is and split **train.csv** into
an 80/20 train/holdout split.


In [ ]:
focused_df = train_df.copy()

train_split_df, holdout_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df["label-fine"],
)
train_split_df = train_split_df.reset_index(drop=True)
holdout_split_df = holdout_split_df.reset_index(drop=True)
test_split_df = test_df.copy().reset_index(drop=True)

print("Train shape:", train_split_df.shape)
print("Holdout shape:", holdout_split_df.shape)
print("Test shape:", test_split_df.shape)

print("\nTrain fine-label distribution:")
display(train_split_df["label-fine"].value_counts().sort_index())

print("\nHoldout fine-label distribution:")
display(holdout_split_df["label-fine"].value_counts().sort_index())

print("\nTest fine-label distribution:")
display(test_split_df["label-fine"].value_counts().sort_index())


In [ ]:
# Dataset label counts (coarse and fine)
datasets = {
    "Train": train_split_df,
    "Holdout": holdout_split_df,
    "Test": test_split_df,
}

for name, df in datasets.items():
    print(f"===== {name} dataset =====")
    print("Coarse label counts:")
    display(df["label-coarse"].value_counts().sort_index())
    print("\nFine label counts:")
    display(df["label-fine"].value_counts().sort_index())
    print()


## Step 3: Choose two embedding models

We still use sentence embeddings, but now the embedding becomes the **input vector** `x in R^d` for the linear supervised learner.


In [ ]:
# Combined dataset label counts (coarse and fine)
combined_df = pd.concat([train_split_df, holdout_split_df, test_split_df], ignore_index=True)

print("===== Combined dataset =====")
print("Coarse label counts:")
display(combined_df["label-coarse"].value_counts().sort_index())
print("\nFine label counts:")
display(combined_df["label-fine"].value_counts().sort_index())


In [ ]:
embedding_models = [
    "all-MiniLM-L6-v2",
    "paraphrase-MiniLM-L6-v2",
]

X_train_text = train_split_df["text"].tolist()
X_test_text = test_split_df["text"].tolist()


y_train_labels = train_split_df["label-fine"].to_numpy()
y_test_labels = test_split_df["label-fine"].to_numpy()


y_test_coarse = test_split_df["label-coarse"].to_numpy()

class_names = sorted(focused_df["label-fine"].unique().tolist())
class_to_idx = {label: idx for idx, label in enumerate(class_names)}
idx_to_class = {idx: label for label, idx in class_to_idx.items()}

class_names_coarse = sorted(focused_df["label-coarse"].unique().tolist())
fine_to_coarse = (
    focused_df[["label-fine", "label-coarse"]]
    .drop_duplicates()
    .set_index("label-fine")["label-coarse"]
    .to_dict()
)

print("Fine classes:", class_names)
print("Number of fine classes:", len(class_names))
print("Coarse classes:", class_names_coarse)
print("Number of coarse classes:", len(class_names_coarse))


In [ ]:
# # Embedding dimensions for each model (no full encoding yet).
# for model_name in embedding_models:
#     model = SentenceTransformer(model_name)
#     embedding_dim = model.get_sentence_embedding_dimension()
#     embedding_matrix_shape = (len(X_train_text), embedding_dim)
#     single_embedding_shape = (embedding_dim,)
#     print(f"Model: {model_name}")
#     print(f"  Embedding matrix shape (train): {embedding_matrix_shape}")
#     print(f"  Single embedding shape: {single_embedding_shape}")


## Helper functions


In [ ]:
def embed_texts(model_name, texts):
    # Load the sentence-transformer model and encode the input texts.
    model = SentenceTransformer(model_name)
    # Encode all texts into a 2D numpy array of embeddings.
    embeddings = model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    return embeddings


def labels_to_index(labels, class_to_idx):
    # Map each label string to its integer class index.
    return np.array([class_to_idx[label] for label in labels], dtype=np.int64)


def one_hot_encode(label_indices, num_classes):
    # Build one-hot rows for each class index.
    return np.eye(num_classes, dtype=np.float32)[label_indices]


def make_multiclass_tensors(X_np, y_labels, class_to_idx):
    # Convert labels to indices and one-hot encode them for training.
    y_idx = labels_to_index(y_labels, class_to_idx)
    y_onehot = one_hot_encode(y_idx, len(class_to_idx))

    # Convert embeddings and one-hot labels into float tensors.
    X_tensor = torch.tensor(X_np, dtype=torch.float32).reshape(X_np.shape)
    Y_onehot_tensor = torch.tensor(y_onehot, dtype=torch.float32).reshape(y_onehot.shape)

    # Return tensors plus raw class indices for metrics.
    return X_tensor, Y_onehot_tensor, y_idx


def model_fit_pytorch(x_train, y_train, model, loss_criterion=None, epochs=1000, lr=0.01):
    # Default to MSE loss for one-hot regression-style training.
    if loss_criterion is None:
        loss_criterion = nn.MSELoss()
    # Use simple SGD for optimization.
    optimizer = optim.SGD(model.parameters(), lr=lr)

    # Track training loss over epochs for plotting.
    loss_history = []

    # Standard training loop: forward, loss, backward, step.
    for epoch in range(epochs):
        model.train()
        optimizer.zero_grad()

        # Forward pass and loss computation.
        y_pred = model(x_train)
        loss = loss_criterion(y_pred, y_train)

        # Backpropagation and parameter update.
        loss.backward()
        optimizer.step()

        # Store loss for later visualization.
        loss_history.append(loss.item())

        if (epoch + 1) % 200 == 0:
            print(f"Epoch {epoch + 1}/{epochs} - loss: {loss.item():.6f}")

    return model, loss_history


def pytorch_model_multiclass_inference(model, X_tensor):
    # Run the model in eval mode and return scores plus class indices.
    model.eval()
    with torch.no_grad():
        # Raw model outputs for each class.
        Y_pred = model(X_tensor)
        # Take argmax to get predicted class index.
        pred_idx = torch.argmax(Y_pred, dim=1).detach().cpu().numpy()
    return Y_pred.detach().cpu().numpy(), pred_idx



def plot_conf_mat(y_true, y_pred, labels, title, save_path=None, cm_override=None):
    # Compute a row-normalized confusion matrix in fractions unless provided.
    if cm_override is None:
        cm = confusion_matrix(y_true, y_pred, labels=labels, normalize="true")
    else:
        cm = cm_override

    # Create a heatmap-style plot.
    fig, ax = plt.subplots(figsize=(max(7, 0.6 * len(labels)), max(6, 0.55 * len(labels))))
    im = ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Label axes with class names.
    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels)
    ax.set_yticklabels(labels)

    # Annotate each cell with its fraction.
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", color="black")

    # Add labels, title, and gridlines for readability.
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True (row fraction)")
    ax.set_title(title)
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right")
    ax.set_xticks(np.arange(len(labels) + 1) - 0.5, minor=True)
    ax.set_yticks(np.arange(len(labels) + 1) - 0.5, minor=True)
    ax.grid(which="minor", color="white", linestyle="-", linewidth=0.5)
    ax.tick_params(which="minor", bottom=False, left=False)
    plt.tight_layout()

    # Optionally save the figure to disk.
    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=200, bbox_inches="tight")

    plt.show()


def build_linear_multiclass_model(input_dim, output_dim):
    # Create a single-layer linear classifier for multiclass prediction.
    return nn.Sequential(nn.Linear(input_dim, output_dim))


def train_one_embedding_pipeline(model_name, X_train_text, X_test_text, y_train_labels, y_test_labels, class_to_idx, class_names, epochs=2000, lr=0.02):
    # End-to-end pipeline: embed text, train classifier, and collect predictions.
    print(f"===== Running {model_name} =====")

    # Generate embeddings for each data split.
    X_train_emb = embed_texts(model_name, X_train_text)
    X_test_emb = embed_texts(model_name, X_test_text)
    # Convert embeddings/labels into PyTorch tensors.
    X_train_tensor, Y_train_onehot_tensor, _ = make_multiclass_tensors(X_train_emb, y_train_labels, class_to_idx)
    X_test_tensor, _, y_test_idx = make_multiclass_tensors(X_test_emb, y_test_labels, class_to_idx)
    # Build a linear classifier sized to the embedding dimension.
    linear_model = build_linear_multiclass_model(
        input_dim=X_train_tensor.shape[1],
        output_dim=Y_train_onehot_tensor.shape[1]
    )

    # Train the model on the training split.
    trained_model, loss_history = model_fit_pytorch(
        X_train_tensor,
        Y_train_onehot_tensor,
        linear_model,
        epochs=epochs,
        lr=lr
    )

    # Predict fine labels on the test split.
    _, test_pred_idx = pytorch_model_multiclass_inference(trained_model, X_test_tensor)
    test_pred_labels = np.array([class_names[i] for i in test_pred_idx], dtype=object)

    # Collect outputs for later evaluation and plotting.
    artifacts = {
        "model_name": model_name,
        "trained_model": trained_model,
        "loss_history": loss_history,
        "y_test_idx": y_test_idx,
        "test_pred_idx": test_pred_idx,
        "test_pred_labels": test_pred_labels,
    }
    return artifacts


## Step 4: Embed the data and train the linear multiclass PyTorch model


In [ ]:
artifacts = {}

for model_name in embedding_models:
    epochs = 10000 if model_name == "paraphrase-MiniLM-L6-v2" else 10000
    lr = 7 if model_name == "paraphrase-MiniLM-L6-v2" else 42
    run_artifacts = train_one_embedding_pipeline(
        model_name=model_name,
        X_train_text=X_train_text,
        X_test_text=X_test_text,
        y_train_labels=y_train_labels,
        y_test_labels=y_test_labels,
        class_to_idx=class_to_idx,
        class_names=class_names,
        epochs=epochs,
        lr=lr,
    )
    artifacts[model_name] = run_artifacts


In [ ]:
# Plot training loss history (log scale) for each embedding model
loss_dir = Path("new_matrices")
loss_dir.mkdir(parents=True, exist_ok=True)

for model_name in embedding_models:
    loss_history = artifacts[model_name]["loss_history"]
    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(len(loss_history)), loss_history)
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Training loss")
    plt.title(f"Loss history — {model_name}")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(loss_dir / f"baseline_{model_name}_loss.png", dpi=200, bbox_inches="tight")
    plt.show()

if "artifacts_augmented" in globals():
    for model_name in embedding_models:
        loss_history = artifacts_augmented[model_name]["loss_history"]
        plt.figure(figsize=(8, 4))
        plt.plot(np.arange(len(loss_history)), loss_history)
        plt.yscale("log")
        plt.xlabel("Epoch")
        plt.ylabel("Training loss")
        plt.title(f"Loss history (augmented) — {model_name}")
        plt.grid(True)
        plt.tight_layout()
        plt.savefig(loss_dir / f"augmented_{model_name}_loss.png", dpi=200, bbox_inches="tight")
        plt.show()


## Step 5: Baseline head-to-head comparison

Now that both embedding models have been trained using the same linear PyTorch classifier pipeline, we compare them in several ways:

- overall metrics on train / test
- fine-label confusion matrices
- per-class F1 differences
- disagreement examples on the same test questions

This lets us separate **overall performance** from **where each embedding succeeds or fails**.



In [ ]:
matrices_dir = Path("matrices")

for model_name in embedding_models:
    plot_conf_mat(
        artifacts[model_name]["y_test_idx"],
        artifacts[model_name]["test_pred_idx"],
        labels=np.arange(len(class_names)),
        title=f"Test confusion matrix (fine) — {model_name}",
        save_path=matrices_dir / f"baseline_{model_name}_fine.png"
    )

    # coarse view: map predicted fine label -> coarse label
    test_pred_coarse = pd.Series(artifacts[model_name]["test_pred_labels"]).map(fine_to_coarse).to_numpy()

    plot_conf_mat(
        y_test_coarse,
        test_pred_coarse,
        labels=class_names_coarse,
        title=f"Test confusion matrix (coarse) — {model_name}",
        save_path=matrices_dir / f"baseline_{model_name}_coarse.png"
    )

In [ ]:
# Download All matrices
from pathlib import Path
import shutil

src_dir = Path('matrices')
dst_dir = Path('new_matrices')
dst_dir.mkdir(parents=True, exist_ok=True)

# newCodeBase writes baseline_{model}_fine/coarse.png -> 4 files total
patterns = ["baseline_*_fine.png", "baseline_*_coarse.png"]
copied = 0
for pattern in patterns:
    for img_path in src_dir.glob(pattern):
        shutil.copy2(img_path, dst_dir / img_path.name)
        copied += 1

print(f'Copied {copied} files to {dst_dir}')


In [ ]:
# Combine the 4 matrices into a single PDF
from pathlib import Path
from PIL import Image

src_dir = Path('new_matrices')
pdf_path = src_dir / 'combined_matrices.pdf'

image_paths = sorted(src_dir.glob('baseline_*_*.png'))
if not image_paths:
    raise FileNotFoundError('No matrices found in new_matrices')

images = [Image.open(p).convert('RGB') for p in image_paths]
images[0].save(pdf_path, save_all=True, append_images=images[1:])

print(f'Wrote {pdf_path} with {len(images)} pages')


In [ ]:
# Compare accuracy scores for each model on fine and coarse labels
accuracy_rows = []
for model_name in embedding_models:
    fine_acc = accuracy_score(
        artifacts[model_name]["y_test_idx"],
        artifacts[model_name]["test_pred_idx"]
    )
    test_pred_coarse = pd.Series(artifacts[model_name]["test_pred_labels"]).map(fine_to_coarse).to_numpy()
    coarse_acc = accuracy_score(y_test_coarse, test_pred_coarse)
    accuracy_rows.append({
        "model": model_name,
        "fine_accuracy": fine_acc,
        "coarse_accuracy": coarse_acc,
    })

accuracy_df = pd.DataFrame(accuracy_rows).sort_values("fine_accuracy", ascending=False)
accuracy_df


## Step 9: Add targeted holdout samples back into training

We upsample the coarse labels 1, 2, and 5 by moving 10% of each class
from the holdout set into the training set. This keeps the test set unchanged
and reduces the holdout size accordingly.


In [ ]:
# Add 30% more samples (per coarse label) from holdout into training.
desired_increase_percentage = 0.3
target_coarse_labels = [1,5]
train_coarse_counts = train_split_df["label-coarse"].value_counts()

added_rows = []
for coarse_label in target_coarse_labels:
    desired_increase = int(np.ceil(train_coarse_counts.get(coarse_label, 0) * desired_increase_percentage))
    if desired_increase <= 0:
        continue
    candidates = holdout_split_df[holdout_split_df["label-coarse"] == coarse_label]
    take_n = min(desired_increase, len(candidates))
    if take_n == 0:
        continue
    added_rows.append(candidates.sample(n=take_n, random_state=RANDOM_STATE))

if added_rows:
    added_df = pd.concat(added_rows)
else:
    added_df = holdout_split_df.head(0).copy()

train_split_df = pd.concat([train_split_df, added_df], ignore_index=True).reset_index(drop=True)
holdout_split_df = holdout_split_df.drop(added_df.index).reset_index(drop=True)

print("Added rows by coarse label:")
display(added_df["label-coarse"].value_counts().sort_index())

print("\nUpdated train coarse-label distribution:")
display(train_split_df["label-coarse"].value_counts().sort_index())

print("\nUpdated holdout coarse-label distribution:")
display(holdout_split_df["label-coarse"].value_counts().sort_index())


## Step 10: Retrain after holdout augmentation

We retrain the embedding + linear model pipeline using the updated training set
and re-evaluate on the original test set.


In [ ]:
X_train_text = train_split_df["text"].tolist()
y_train_labels = train_split_df["label-fine"].to_numpy()

artifacts_augmented = {}

for model_name in embedding_models:
    epochs = 10000 if model_name == "paraphrase-MiniLM-L6-v2" else 20000
    lr = 7 if model_name == "paraphrase-MiniLM-L6-v2" else 40
    run_artifacts = train_one_embedding_pipeline(
        model_name=model_name,
        X_train_text=X_train_text,
        X_test_text=X_test_text,
        y_train_labels=y_train_labels,
        y_test_labels=y_test_labels,
        class_to_idx=class_to_idx,
        class_names=class_names,
        epochs=epochs,
        lr=lr,
    )
    artifacts_augmented[model_name] = run_artifacts


In [ ]:
# Plot training loss history (log scale) for augmented models
loss_dir = Path("new_matrices")
loss_dir.mkdir(parents=True, exist_ok=True)

for model_name in embedding_models:
    loss_history = artifacts_augmented[model_name]["loss_history"]
    plt.figure(figsize=(8, 4))
    plt.plot(np.arange(len(loss_history)), loss_history)
    plt.yscale("log")
    plt.xlabel("Epoch")
    plt.ylabel("Training loss")
    plt.title(f"Loss history (augmented) — {model_name}")
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(loss_dir / f"augmented_{model_name}_loss.png", dpi=200, bbox_inches="tight")
    plt.show()


## Step 11: Confusion matrices after holdout augmentation

We visualize fine and coarse confusion matrices for the retrained models.


In [ ]:
# Recompute accuracy after augmentation
accuracy_rows = []
for model_name in embedding_models:
    fine_acc = accuracy_score(
        artifacts_augmented[model_name]["y_test_idx"],
        artifacts_augmented[model_name]["test_pred_idx"]
    )
    test_pred_coarse = pd.Series(artifacts_augmented[model_name]["test_pred_labels"]).map(fine_to_coarse).to_numpy()
    coarse_acc = accuracy_score(y_test_coarse, test_pred_coarse)
    accuracy_rows.append({
        "model": model_name,
        "fine_accuracy": fine_acc,
        "coarse_accuracy": coarse_acc,
    })

accuracy_df_augmented = pd.DataFrame(accuracy_rows).sort_values("fine_accuracy", ascending=False)
accuracy_df_augmented

In [ ]:
matrices_dir = Path("new_matrices")
matrices_dir.mkdir(parents=True, exist_ok=True)

for model_name in embedding_models:
    plot_conf_mat(
        artifacts_augmented[model_name]["y_test_idx"],
        artifacts_augmented[model_name]["test_pred_idx"],
        labels=np.arange(len(class_names)),
        title=f"Test confusion matrix (fine, augmented) — {model_name}",
        save_path=matrices_dir / f"augmented_{model_name}_fine.png"
    )

    test_pred_coarse = pd.Series(artifacts_augmented[model_name]["test_pred_labels"]).map(fine_to_coarse).to_numpy()

    plot_conf_mat(
        y_test_coarse,
        test_pred_coarse,
        labels=class_names_coarse,
        title=f"Test confusion matrix (coarse, augmented) — {model_name}",
        save_path=matrices_dir / f"augmented_{model_name}_coarse.png"
    )


In [ ]:
# Random 5-label fine confusion matrices (augmented)
if "rng" not in globals():
    rng = np.random.default_rng(RANDOM_STATE)
subset_labels = rng.choice(class_names, size=6, replace=False).tolist()
subset_labels = sorted(subset_labels)
print("Subset fine labels:", subset_labels)

subset_indices = [class_to_idx[label] for label in subset_labels]

for model_name in embedding_models:
    y_true_labels = np.asarray(y_test_labels)
    test_pred_idx = artifacts_augmented[model_name]["test_pred_idx"]
    y_pred_labels = np.take(class_names, test_pred_idx)

    cm_full = confusion_matrix(
        y_true_labels,
        y_pred_labels,
        labels=class_names,
        normalize="true",
    )
    cm_subset = cm_full[np.ix_(subset_indices, subset_indices)]

    plot_conf_mat(
        y_true_labels,
        y_pred_labels,
        labels=subset_labels,
        title=f"Test confusion matrix (fine, augmented, 5-label subset) — {model_name}",
        save_path=Path("new_matrices") / f"augmented_{model_name}_fine_subset5.png",
        cm_override=cm_subset,
    )


In [ ]:
# Copy confusion matrices into new_matrices
from pathlib import Path
import shutil

src_dir = Path("matrices")
dst_dir = Path("new_matrices")
dst_dir.mkdir(parents=True, exist_ok=True)

copied = 0
for img_path in src_dir.glob("*.png"):
    shutil.copy2(img_path, dst_dir / img_path.name)
    copied += 1

print(f"Copied {copied} files to {dst_dir}")


In [ ]:
# Copy lr/epoch grid images into new_matrices
from pathlib import Path
import shutil

src_dir = Path("new_matrices")
dst_dir = Path("new_matrices")
dst_dir.mkdir(parents=True, exist_ok=True)

copied = 0
skipped = 0
for img_path in src_dir.glob("lr_epoch_accuracy_*.png"):
    dest_path = dst_dir / img_path.name
    if img_path.resolve() == dest_path.resolve():
        skipped += 1
        continue
    shutil.copy2(img_path, dest_path)
    copied += 1

print(f"Copied {copied} grid files to {dst_dir} (skipped {skipped} already there)")


In [ ]:
# Fine-label subset confusion matrices (top/bottom 10 by train frequency)
train_counts = train_split_df["label-fine"].value_counts()

top10_labels = train_counts.head(10).index.tolist()
bottom10_labels = train_counts.sort_values(ascending=True).head(10).index.tolist()

subset_groups = {
    "fine_top10": top10_labels,
    "fine_bottom10": bottom10_labels,
}

artifacts_source = artifacts_augmented if "artifacts_augmented" in globals() else artifacts

for subset_name, subset_labels in subset_groups.items():
    print(f"{subset_name} labels: {subset_labels}")
    subset_indices = [class_to_idx[label] for label in subset_labels]

    for model_name in embedding_models:
        y_true_labels = np.asarray(y_test_labels)
        test_pred_idx = artifacts_source[model_name]["test_pred_idx"]
        y_pred_labels = np.take(class_names, test_pred_idx)

        cm_full = confusion_matrix(
            y_true_labels,
            y_pred_labels,
            labels=class_names,
            normalize="true",
        )
        cm_subset = cm_full[np.ix_(subset_indices, subset_indices)]

        plot_conf_mat(
            y_true_labels,
            y_pred_labels,
            labels=subset_labels,
            title=f"Test confusion matrix ({subset_name}) — {model_name}",
            save_path=Path("new_matrices") / f"{subset_name}_{model_name}.png",
            cm_override=cm_subset,
        )


In [ ]:
# Bar chart for all fine-label counts with accuracy annotations (combined models)
train_counts = train_split_df["label-fine"].value_counts().sort_values(ascending=False)

labels = train_counts.index.tolist()
counts = train_counts.values.tolist()

artifacts_source = artifacts_augmented if "artifacts_augmented" in globals() else artifacts
y_true_labels = np.asarray(y_test_labels)

accuracy_by_model = {}
combined_cm = None
for model_name in embedding_models:
    test_pred_idx = artifacts_source[model_name]["test_pred_idx"]
    y_pred_labels = np.take(class_names, test_pred_idx)

    cm_full = confusion_matrix(
        y_true_labels,
        y_pred_labels,
        labels=class_names,
        normalize="true",
    )

    accuracy_by_model[model_name] = np.array(
        [cm_full[class_to_idx[label], class_to_idx[label]] for label in labels]
    )

    if combined_cm is None:
        combined_cm = cm_full
    else:
        combined_cm += cm_full

combined_cm = combined_cm / max(len(embedding_models), 1)

combined_acc = np.zeros(len(labels), dtype=float)
for acc_values in accuracy_by_model.values():
    combined_acc += acc_values

if combined_acc.max() > combined_acc.min():
    combined_acc_norm = (combined_acc - combined_acc.min()) / (combined_acc.max() - combined_acc.min())
else:
    combined_acc_norm = np.zeros_like(combined_acc)

cmap = plt.cm.viridis
colors = cmap(combined_acc_norm)

fig, ax_counts = plt.subplots(figsize=(14, 6))
x = np.arange(len(labels))

ax_counts.bar(x, counts, color=colors, label="Train count")

ax_counts.set_xticks(x)
ax_counts.set_xticklabels([str(label) for label in labels], rotation=60, ha="right")
ax_counts.set_ylabel("Training count")
ax_counts.set_title("Fine-label counts with accuracy (both models)")
ax_counts.grid(axis="y", linestyle="--", alpha=0.5)

sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(vmin=combined_acc_norm.min(), vmax=combined_acc_norm.max()))
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax_counts, pad=0.01)
cbar.set_label("Normalized sum of accuracies")

ax_counts.legend(loc="upper right")

fig.tight_layout()
plt.show()

plot_conf_mat(
    y_true_labels,
    y_true_labels,
    labels=class_names,
    title="Combined test confusion matrix (fine, normalized sum)",
    cm_override=combined_cm,
)


In [ ]:
# # Learning rate vs epoch accuracy grids (per model)
# lr_values = [0.1, 0.5, 1, 7, 20, 42, 44]
# epoch_values = [5000, 10000]

# # Note: this trains len(lr_values) * len(epoch_values) models per embedding model.
# def compute_accuracy_grid_for_model(
#     model_name,
#     X_train_text,
#     X_test_text,
#     y_train_labels,
#     y_test_labels,
#     class_to_idx,
#     class_names,
#     lr_values,
#     epoch_values,
# ):
#     X_train_emb = embed_texts(model_name, X_train_text)
#     X_test_emb = embed_texts(model_name, X_test_text)

#     X_train_tensor, Y_train_onehot_tensor, _ = make_multiclass_tensors(
#         X_train_emb,
#         y_train_labels,
#         class_to_idx,
#     )
#     X_test_tensor, _, y_test_idx = make_multiclass_tensors(
#         X_test_emb,
#         y_test_labels,
#         class_to_idx,
#     )

#     input_dim = X_train_tensor.shape[1]
#     output_dim = Y_train_onehot_tensor.shape[1]

#     acc_grid = np.zeros((len(lr_values), len(epoch_values)), dtype=float)

#     for i, lr in enumerate(lr_values):
#         for j, epochs in enumerate(epoch_values):
#             model = build_linear_multiclass_model(input_dim, output_dim)
#             trained_model, _ = model_fit_pytorch(
#                 X_train_tensor,
#                 Y_train_onehot_tensor,
#                 model,
#                 epochs=epochs,
#                 lr=lr,
#             )
#             _, pred_idx = pytorch_model_multiclass_inference(trained_model, X_test_tensor)
#             acc_grid[i, j] = accuracy_score(y_test_idx, pred_idx)

#     return acc_grid


# def plot_accuracy_grid_table(acc_grid, lr_values, epoch_values, title, save_path=None):
#     epoch_labels = [f"{e // 1000}k" for e in epoch_values]
#     cell_text = [["lr/epochs"] + epoch_labels]
#     for lr, row in zip(lr_values, acc_grid):
#         cell_text.append([f"{lr:.1f}"] + [f"{acc:.2f}" for acc in row])

#     fig, ax = plt.subplots(figsize=(6, 6))
#     ax.axis("off")
#     table = ax.table(cellText=cell_text, cellLoc="center", loc="center")
#     table.auto_set_font_size(False)
#     table.set_fontsize(10)
#     table.scale(1, 1.4)
#     ax.set_title(title)
#     fig.tight_layout()

#     if save_path is not None:
#         save_path = Path(save_path)
#         save_path.parent.mkdir(parents=True, exist_ok=True)
#         fig.savefig(save_path, dpi=200, bbox_inches="tight")

#     plt.show()


# for model_name in embedding_models:
#     acc_grid = compute_accuracy_grid_for_model(
#         model_name,
#         X_train_text,
#         X_test_text,
#         y_train_labels,
#         y_test_labels,
#         class_to_idx,
#         class_names,
#         lr_values,
#         epoch_values,
#     )

#     plot_accuracy_grid_table(
#         acc_grid,
#         lr_values,
#         epoch_values,
#         title=f"Accuracy grid — {model_name}",
#         save_path=Path("new_matrices") / f"lr_epoch_accuracy_{model_name}.png",
#     )


## Step 12: Random 9-per-coarse tester

Evaluate each model on a random subset of 9 test entries per coarse label.


In [ ]:
# Random 9-per-coarse tester
if "rng" not in globals():
    rng = np.random.default_rng(RANDOM_STATE)

artifacts_source = artifacts_augmented if "artifacts_augmented" in globals() else artifacts
source_df = test_split_df.copy()

sampled_rows = []
for coarse_label in sorted(source_df["label-coarse"].unique()):
    group = source_df[source_df["label-coarse"] == coarse_label]
    if group.empty:
        continue
    n = min(9, len(group))
    sampled_rows.append(
        group.sample(n=n, random_state=int(rng.integers(0, 1_000_000)))
    )

sampled_df = pd.concat(sampled_rows, ignore_index=True)
print("Random subset size:", len(sampled_df))
display(sampled_df["label-coarse"].value_counts().sort_index())

X_sample_text = sampled_df["text"].tolist()
y_sample_labels = sampled_df["label-fine"].to_numpy()
y_sample_coarse = sampled_df["label-coarse"].to_numpy()

for model_name in embedding_models:
    print(f"===== Random 9-per-coarse test: {model_name} =====")
    X_sample_emb = embed_texts(model_name, X_sample_text)
    X_sample_tensor, _, y_sample_idx = make_multiclass_tensors(
        X_sample_emb,
        y_sample_labels,
        class_to_idx,
    )
    _, pred_idx = pytorch_model_multiclass_inference(
        artifacts_source[model_name]["trained_model"],
        X_sample_tensor,
    )
    pred_labels = np.array([class_names[i] for i in pred_idx], dtype=object)
    pred_coarse = pd.Series(pred_labels).map(fine_to_coarse).to_numpy()

    fine_acc = accuracy_score(y_sample_idx, pred_idx)
    coarse_acc = accuracy_score(y_sample_coarse, pred_coarse)
    print(f"Fine accuracy (subset): {fine_acc:.3f}")
    print(f"Coarse accuracy (subset): {coarse_acc:.3f}")

    results_df = pd.DataFrame({
        "text": sampled_df["text"].tolist(),
        "true_fine": y_sample_labels,
        "pred_fine": pred_labels,
        "true_coarse": y_sample_coarse,
        "pred_coarse": pred_coarse,
    })
    display(results_df)


In [ ]:
# Incrementally add 10% of remaining holdout data before each retrain
# NOTE: Each round retrains both models; reduce rounds/epochs for faster runs.
holdout_add_fraction = 0.10
max_rounds = 5

# Rebuild the base split so each round starts from the same point.
base_train_split_df, base_holdout_split_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=train_df["label-fine"],
)
base_train_split_df = base_train_split_df.reset_index(drop=True)
base_holdout_split_df = base_holdout_split_df.reset_index(drop=True)

train_current = base_train_split_df.copy()
holdout_remaining = base_holdout_split_df.copy()

if "rng" not in globals():
    rng = np.random.default_rng(RANDOM_STATE)

round_results = []
artifacts_by_round = {}

for round_idx in range(1, max_rounds + 1):
    if holdout_remaining.empty:
        print("Holdout is empty; stopping early.")
        break

    add_n = int(np.ceil(len(holdout_remaining) * holdout_add_fraction))
    add_n = max(add_n, 1)
    added_df = holdout_remaining.sample(
        n=add_n,
        random_state=int(rng.integers(0, 1_000_000)),
    )

    train_current = pd.concat([train_current, added_df], ignore_index=True).reset_index(drop=True)
    holdout_remaining = holdout_remaining.drop(added_df.index).reset_index(drop=True)

    X_train_text = train_current["text"].tolist()
    y_train_labels = train_current["label-fine"].to_numpy()

    round_artifacts = {}

    for model_name in embedding_models:
        epochs = 10000 if model_name == "paraphrase-MiniLM-L6-v2" else 20000
        lr = 7 if model_name == "paraphrase-MiniLM-L6-v2" else 40
        run_artifacts = train_one_embedding_pipeline(
            model_name=model_name,
            X_train_text=X_train_text,
            X_test_text=X_test_text,
            y_train_labels=y_train_labels,
            y_test_labels=y_test_labels,
            class_to_idx=class_to_idx,
            class_names=class_names,
            epochs=epochs,
            lr=lr,
        )
        round_artifacts[model_name] = run_artifacts

        fine_acc = accuracy_score(
            run_artifacts["y_test_idx"],
            run_artifacts["test_pred_idx"],
        )
        test_pred_coarse = pd.Series(run_artifacts["test_pred_labels"]).map(fine_to_coarse).to_numpy()
        coarse_acc = accuracy_score(y_test_coarse, test_pred_coarse)

        round_results.append({
            "round": round_idx,
            "model": model_name,
            "train_size": len(train_current),
            "holdout_size": len(holdout_remaining),
            "added_rows": len(added_df),
            "fine_accuracy": fine_acc,
            "coarse_accuracy": coarse_acc,
        })

    artifacts_by_round[round_idx] = round_artifacts

round_results_df = pd.DataFrame(round_results).sort_values(["round", "model"])
round_results_df
